In [120]:
import numpy as np
from collections import deque

In [155]:
np.random.seed(42)

In [ ]:
class NeuralNetwork:
    def __init__(self, layer_neurons: list[int]):
        self.lin_layers = list()
        self.bias = list()
        self.activations = list()
        self.lin_gradients = deque()
        self.bias_gradients = deque()
        self.y_true = None
        for i in range(len(layer_neurons)):
            if i == 0:
                continue
            self.lin_layers.append(np.random.randn(layer_neurons[i - 1], layer_neurons[i]))
            self.bias.append(np.random.randn(1, layer_neurons[i]))

    def sigmoid(self, x: np.array):
        return 1 / (1 + np.exp(-x))

    def cross_entropy(self, y_pred, y_true):
        return y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred)

    # def cross_entropy_gradient(self, y_pred, y_true):
    #     return (y_pred - y_true) / (y_pred - y_pred**2)

    def cross_entropy_gradient(self, y_pred, y_true):
        # Add numerical stability and normalize by batch size
        epsilon = 1e-15
        y_pred = np.clip(y_pred, epsilon, 1 - epsilon)
        batch_size = y_pred.shape[0]
        return (y_pred - y_true) / (y_pred * (1 - y_pred)) / batch_size

    def forward(self, x, y_true=None):
        x = np.atleast_2d(x)
        self.y_true = y_true
        store_activations = y_true != None
        for layer, bias in zip(self.lin_layers, self.bias):
            if store_activations:
                self.activations.append(x)
            x = self.sigmoid(x @ layer + bias)
        if store_activations:
            self.activations.append(x)
        return x

    def sigmoid_gradient(self, x):
        return x * (1 - x)

    def backward(self):
        next_layer_activation_gradient = self.cross_entropy_gradient(
            self.activations[-1], self.y_true
        )
        for i in range(len(self.lin_layers) - 1, -1, -1):
            current_activation = self.activations[i]
            next_activation = self.activations[i + 1]
            pre_act_grad = self.sigmoid_gradient(next_activation) * next_layer_activation_gradient
            lin_grad = current_activation.T @ pre_act_grad
            self.lin_gradients.appendleft(lin_grad)
            self.bias_gradients.appendleft(pre_act_grad)
            next_layer_activation_gradient = pre_act_grad @ self.lin_layers[i].T

    def step(self, lr=1e-2):
        for i in range(len(self.lin_layers)):
            self.lin_layers[i] -= lr * self.lin_gradients[i]
            self.bias[i] -= lr * self.bias_gradients[i]

    def clear_grad(self):
        self.lin_gradients = deque()
        self.bias_gradients = deque()

In [595]:
nn = NeuralNetwork([2, 3, 1])
inputs = np.array(
    [
        [0, 0],
        [1, 0],
        [0, 1],
        [1, 1],
    ]
)

In [618]:
for i in range(10_000):
    input = inputs[np.random.choice(inputs.shape[0])]
    y = np.bitwise_xor(*input)
    r = nn.forward(input, y)
    # if not i % 1_000:
    # print(i, y, r)
    nn.backward()
    nn.step(lr=1e-3)
    nn.clear_grad()

for i in inputs:
    print(i, nn.forward(i), np.bitwise_xor(*i))

[0 0] [[0.59066335]] 0
[1 0] [[0.47182456]] 1
[0 1] [[0.51379877]] 1
[1 1] [[0.39682537]] 0


In [ ]:
class NeuralNetwork:
    def __init__(self, layer_neurons: list[int]):
        self.lin_layers = list()
        self.bias = list()
        self.activations = list()
        self.lin_gradients = deque()
        self.bias_gradients = deque()
        self.y_true = None
        for i in range(len(layer_neurons)):
            if i == 0:
                continue
            self.lin_layers.append(np.random.randn(layer_neurons[i - 1], layer_neurons[i]))
            self.bias.append(np.random.randn(1, layer_neurons[i]))

    def sigmoid(self, x: np.array):
        return 1 / (1 + np.exp(-x))

    # def cross_entropy(self, y_pred, y_true):
    #     # Add numerical stability and handle batch dimension
    #     epsilon = 1e-15
    #     y_pred = np.clip(y_pred, epsilon, 1 - epsilon)
    #     batch_loss = y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred)
    #     return -np.mean(batch_loss)

    def cross_entropy_gradient(self, y_pred, y_true):
        # Add numerical stability and normalize by batch size
        epsilon = 1e-15
        y_pred = np.clip(y_pred, epsilon, 1 - epsilon)
        batch_size = y_pred.shape[0]
        return (y_pred - y_true) / (y_pred * (1 - y_pred)) / batch_size

    # def cross_entropy_gradient(self, y_pred, y_true):
    #     return (y_pred - y_true) / (y_pred - y_pred**2)

    def forward(self, x, y_true=None):
        x = np.atleast_2d(x)
        self.y_true = y_true
        store_activations = y_true is not None

        # Clear previous activations for batch processing
        if store_activations:
            self.activations = list()

        for layer, bias in zip(self.lin_layers, self.bias):
            if store_activations:
                self.activations.append(x)
            x = self.sigmoid(x @ layer + bias)
        if store_activations:
            self.activations.append(x)
        return x

    def sigmoid_gradient(self, x):
        return x * (1 - x)

    def backward(self):
        next_layer_activation_gradient = self.cross_entropy_gradient(
            self.activations[-1], self.y_true
        )

        for i in range(len(self.lin_layers) - 1, -1, -1):
            current_activation = self.activations[i]
            next_activation = self.activations[i + 1]
            pre_act_grad = self.sigmoid_gradient(next_activation) * next_layer_activation_gradient

            lin_grad = current_activation.T @ pre_act_grad
            bias_grad = np.sum(pre_act_grad, axis=0, keepdims=True)

            self.lin_gradients.appendleft(lin_grad)
            self.bias_gradients.appendleft(bias_grad)
            next_layer_activation_gradient = pre_act_grad @ self.lin_layers[i].T

    def step(self, lr=1e-2):
        for i in range(len(self.lin_layers)):
            self.lin_layers[i] -= lr * self.lin_gradients[i]
            self.bias[i] -= lr * self.bias_gradients[i]

    def clear_grad(self):
        self.lin_gradients = deque()
        self.bias_gradients = deque()

In [627]:
nn = NeuralNetwork([2, 3, 1])
inputs = np.array(
    [
        [0, 0],
        [1, 0],
        [0, 1],
        [1, 1],
    ]
)

In [628]:
for i in inputs:
    r = nn.forward(i)
    print(i, r, np.bitwise_xor(*i))

[0 0] [[0.76460267]] 0
[1 0] [[0.78983175]] 1
[0 1] [[0.74075616]] 1
[1 1] [[0.76770118]] 0


In [633]:
for i in range(10_000):
    y = np.array([[np.bitwise_xor(*input)] for input in inputs])
    r = nn.forward(inputs, y)
    nn.backward()
    nn.step(lr=1e-2)
    nn.clear_grad()

for i in inputs:
    r = nn.forward(i)
    print(i, r, np.bitwise_xor(*i))

[0 0] [[0.01941842]] 0
[1 0] [[0.93932823]] 1
[0 1] [[0.9267313]] 1
[1 1] [[0.0955381]] 0
